In [1]:
import json
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

import config
from input.input import load_raw_data
from model import generics, single_ml_model_exp, grid_search_exp
from sklearn.neural_network import MLPRegressor

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen impor

In [2]:
# === Notebook de BASELINE na janela de 10% (pct10) -- MLP single (SKlearnModel) ===
# Janela: lag_size = resolve_lag_size_pct(N - test_size, 0.10) por serie
# (base identica a get_max_lag_to_consider). NAO e o baseline 'auto':
# experiment_id='chamados_pct10' e model_name='mlppct10' sao
# distintos e nunca colidem com a matriz 'auto' ja validada. Modelo NU
# (sem seletor). force=False. Nomenclatura sem underscore (RUNBOOK.md 7).
# Unica acao humana: Restart Kernel -> Run All.
model = MLPRegressor(activation='logistic', solver='lbfgs')

series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt', 'windspeedfortaleza.txt', 'samurec.txt']

experiment_id = 'chamados_pct10'
model_name = 'mlppct10'   # -> 1mlppct10.pkl
normalize = True
force = False
model_exec = 10

experiment_params = {
    'diff_kpss': False,
    'horizon': 1,
    'type_filter': None,
}

model_parameters = {
    'hidden_layer_sizes': [10, 20, 50],
    'max_iter': [1000],
}

experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

In [3]:
# Janela de 10%: lag_size_override = resolve_lag_size_pct(N - test_size, 0.10)
# por serie -- MESMA base que get_max_lag_to_consider (PACF sobre
# ts_univariate[0:-test_size]). Computado aqui, nada a editar.
# force=False: execution() (correcao de 2026-09-02) pula .pkl ja existente
# e nao-vazio -- re-Run All e idempotente.
lag_pct_por_serie = {}
for base_name in series_list:
    n_raw = len(load_raw_data(base_name))
    n_train = n_raw - int(config.TEST_SIZE * n_raw)
    lag_pct = grid_search_exp.resolve_lag_size_pct(n_train, pct=0.10)
    lag_pct_por_serie[base_name] = lag_pct
    print(f'{base_name}  N={n_raw}  N-test={n_train}  lag_pct={lag_pct}')
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
        lag_size_override=lag_pct,
        estimator_random_state_base=grid_search_exp.MLP_RANDOM_STATE_BASE,  # CLAUDE.md 3.4 -- seed fixa das familias MLP
    )
    exec_gs.execution()

airlines.txt  N=144  N-test=130  lag_pct=13
{'hidden_layer_sizes': 10, 'max_iter': 1000}
austres.txt  N=89  N-test=81  lag_pct=8
{'hidden_layer_sizes': 50, 'max_iter': 1000}
coloradoRiver.txt  N=744  N-test=670  lag_pct=67
{'hidden_layer_sizes': 10, 'max_iter': 1000}
sunspot.txt  N=288  N-test=260  lag_pct=26
{'hidden_layer_sizes': 20, 'max_iter': 1000}
windspeedfortaleza.txt  N=144  N-test=130  lag_pct=13
{'hidden_layer_sizes': 20, 'max_iter': 1000}
samurec.txt  N=1188  N-test=1070  lag_pct=107
{'hidden_layer_sizes': 10, 'max_iter': 1000}


In [4]:
from utils.export_metrics_to_csv import run_export_metrics_to_csv

df_metrics = run_export_metrics_to_csv(
    experiment_dir, experiment_dir_results / 'metrics.csv', detail=True,
)
df_metrics

[INFO] 18 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10'.

  OK  airlines_1arima.pkl  ->  1 linha(s)
  OK  airlines_1aspct10.pkl  ->  1 linha(s)
  OK  airlines_1mlppct10.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  1 linha(s)
  OK  austres_1aspct10.pkl  ->  1 linha(s)
  OK  austres_1mlppct10.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  1 linha(s)
  OK  coloradoRiver_1aspct10.pkl  ->  1 linha(s)
  OK  coloradoRiver_1mlppct10.pkl  ->  10 linha(s)
  OK  samurec_1arima.pkl  ->  1 linha(s)
  OK  samurec_1aspct10.pkl  ->  1 linha(s)
  OK  samurec_1mlppct10.pkl  ->  10 linha(s)
  OK  sunspot_1arima.pkl  ->  1 linha(s)
  OK  sunspot_1aspct10.pkl  ->  1 linha(s)
  OK  sunspot_1mlppct10.pkl  ->  10 linha(s)
  OK  windspeedfortaleza_1arima.pkl  ->  1 linha(s)
  OK  windspeedfortaleza_1aspct10.pkl  ->  1 linha(s)
  OK  windspeedfortaleza_1mlppct10.pkl  ->  10 linha(s)

[OK] CSV agregado (média das repetições) gerado em: C:\P

,ExperimentID,Serie,Modelo,N_Repeticoes,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,theil_mean,theil_std,ARV_mean,ARV_std,IA_mean,IA_std,POCID_mean,POCID_std
0,chamados_pct10,airlines,1arima,1,389.194049,NaN,19.728002,NaN,15.102607,NaN,3.315326,NaN,0.134713,NaN,0.066301,NaN,0.983139,NaN,78.571429,NaN
1,chamados_pct10,airlines,1aspct10,1,401.409326,NaN,20.035202,NaN,17.057345,NaN,3.633918,NaN,0.150045,NaN,0.070068,NaN,0.982385,NaN,78.571429,NaN
2,chamados_pct10,airlines,1mlppct10,10,864.535481,378.922193,28.856466,5.947906,24.851256,6.084922,5.313748,1.306045,0.527235,0.205808,0.156302,0.054124,0.960548,0.016086,78.571429,7.529233
3,chamados_pct10,austres,1arima,1,358.130880,NaN,18.924346,NaN,13.710292,NaN,0.078380,NaN,0.129713,NaN,0.034904,NaN,0.990999,NaN,75.000000,NaN
4,chamados_pct10,austres,1aspct10,1,356.337301,NaN,18.876899,NaN,13.900089,NaN,0.079505,NaN,0.134889,NaN,0.035708,NaN,0.990920,NaN,75.000000,NaN
5,chamados_pct10,austres,1mlppct10,10,673.293637,496.477304,24.680016,8.445276,21.594809,8.984090,0.123502,0.051346,0.333874,0.262876,0.052660,0.037981,0.984781,0.011329,87.500000,0.000000
6,chamados_pct10,coloradoRiver,1arima,1,0.104961,NaN,0.323976,NaN,0.262984,NaN,28.817193,NaN,0.755655,NaN,0.698832,NaN,0.667249,NaN,52.702703,NaN
7,chamados_pct10,coloradoRiver,1aspct10,1,0.198910,NaN,0.445994,NaN,0.365586,NaN,40.967593,NaN,0.901369,NaN,0.762988,NaN,0.569918,NaN,60.810811,NaN
8,chamados_pct10,coloradoRiver,1mlppct10,10,0.045050,0.013290,0.210362,0.029779,0.162060,0.025554,20.073847,2.750501,1.212716,0.357258,0.697075,0.092454,0.760135,0.049740,62.837838,2.136674
9,chamados_pct10,samurec,1arima,1,48.481589,NaN,6.962872,NaN,5.858474,NaN,20.580083,NaN,28.356862,NaN,34.562540,NaN,0.193628,NaN,64.406780,NaN


In [5]:
import json

metadata = {
    'experiment_id': experiment_id,
    'notebook': 'single_models/mlp_pct10.ipynb',
    'tipo': 'baseline pct10',
    'familia': 'MLP single (SKlearnModel)',
    'janela': 'pct10 -- resolve_lag_size_pct(N - int(config.TEST_SIZE*N), 0.10)',
    'lag_pct_por_serie': {s: lag_pct_por_serie[s] for s in series_list},
    'series': series_list,
    'model_exec': model_exec,
    'seed': grid_search_exp.MLP_RANDOM_STATE_BASE,
    'model_parameters': model_parameters,
    'diff_kpss': experiment_params['diff_kpss'],
}
experiment_dir_results.mkdir(parents=True, exist_ok=True)
(experiment_dir_results / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, default=str), encoding='utf-8'
)
print('metadata.json ->', experiment_dir_results / 'metadata.json')

Failed to read module file 'C:\Projetos\mestrado_codigos\experiments\src\model\hybrid_system_exp.py' for module 'model.hybrid_system_exp': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 39146: character maps to <undefined>
Failed to read module file 'C:\Projetos\mestrado_codigos\experiments\src\utils\export_metrics_to_csv.py' for module 'utils.export_metrics_to_csv': UnicodeDecodeError
Traceback (most recent call last)

metadata.json -> C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10\metadata.json
